In [ ]:
!pip install -q transformers

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import scipy.io.wavfile
import gc
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display, HTML

# 1. Your new Language Configuration Dictionary
LANG_CONFIG = {
    'English': {'glotlid': ['eng'], 'tar': 'english_audio.tar.gz', 'supports_xtts': True, 'xtts_code': 'en', 'mms_id': 'facebook/mms-tts-eng'},
    'French': {'glotlid': ['fra'], 'tar': 'french_audio.tar.gz', 'supports_xtts': True, 'xtts_code': 'fr', 'mms_id': 'facebook/mms-tts-fra'},
    'German': {'glotlid': ['deu'], 'tar': 'german_audio.tar.gz', 'supports_xtts': True, 'xtts_code': 'de', 'mms_id': 'facebook/mms-tts-deu'},
    'Spanish': {'glotlid': ['spa'], 'tar': 'spanish_audio.tar.gz', 'supports_xtts': True, 'xtts_code': 'es', 'mms_id': 'facebook/mms-tts-spa'},
    'Chinese': {'glotlid': ['zho', 'cmn'], 'tar': 'chinese_audio.tar.gz', 'supports_xtts': True, 'xtts_code': 'zh-cn', 'mms_id': 'facebook/mms-tts-cmn'},
    'Catalan': {'glotlid': ['cat'], 'tar': 'catalan_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-cat'},
    'Esperanto': {'glotlid': ['epo'], 'tar': 'esperanto_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-epo'},
    'Bengali': {'glotlid': ['ben'], 'tar': 'bengali_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-ben'},
    'Belarusian': {'glotlid': ['bel'], 'tar': 'belarusian_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-bel'},
    'Pashto': {'glotlid': ['pus', 'pbt', 'pbu'], 'tar': 'pashto_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-pus'},
    'Kinyarwanda': {'glotlid': ['kin'], 'tar': 'kinyarwanda_audio.tar.gz', 'supports_xtts': False, 'mms_id': 'facebook/mms-tts-kin'}
}

# 2. Sample Sentences
sentences = {
    "French": "Ceci est une voix générée artificiellement par le modèle MMS.",
    "German": "Dies ist eine künstlich erzeugte Stimme durch das MMS-Modell.",
    "Spanish": "Esta es una voz generada artificialmente por el modelo MMS.",
    "English": "That's even how your millennial sibling talks about their childhood.",
    "Kinyarwanda": "Ubu ni ijwi ryakozwe mu buryo bw'ubukorikori hakoreshejwe MMS.",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo de la MMS-modelo.",
    "Catalan": "Aquesta és una veu generada artificialment pel model MMS.",
    "Chinese": "这是由 MMS 模型人工生成的语音。",
    "Bengali": "এটি এমএমএস মডেল দ্বারা কৃত্রিমভাবে তৈরি একটি কণ্ঠস্বর।",
    "Pashto": "دا د ایم ایم ایس ماډل لخوا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас з дапамогай мадэлі MMS."
}

def generate_audio_batch():
    """Loops through the language config, generates audio, and manages Colab memory."""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device.upper()}\n" + "="*40)

    for lang_name, lang_info in LANG_CONFIG.items():
        # Retrieve the exact model ID from your configuration
        model_id = lang_info['mms_id']

        # Get the sentence (skip if we don't have a sentence for this language)
        text = sentences.get(lang_name)
        if not text:
            continue

        output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.wav")

        display(HTML(f"<h3>Processing {lang_name}</h3>"))
        print(f"Targeting model: {model_id}...")

        try:
            # Load tokenizer and model
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = VitsModel.from_pretrained(model_id).to(device)

            # Generate Audio
            inputs = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                output = model(**inputs).waveform

            audio_data = output.squeeze().cpu().numpy()
            sample_rate = model.config.sampling_rate

            # Save the file
            scipy.io.wavfile.write(output_filename, rate=sample_rate, data=audio_data)
            print(f"✅ Success! Saved to: {output_filename}")

            # Display playable audio widget
            display(Audio(output_filename))

        except Exception as e:
            # This will gracefully catch the missing Esperanto, Belarusian, and Pashto models
            print(f"❌ Failed to process {lang_name}. Error: {e}")

        finally:
            # Clear memory to prevent Colab crashes
            if 'model' in locals(): del model
            if 'tokenizer' in locals(): del tokenizer
            if 'inputs' in locals(): del inputs
            if 'output' in locals(): del output

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print("-" * 40)

# Run the batch process
generate_audio_batch()

Using device: CUDA


Targeting model: facebook/mms-tts-eng...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

✅ Success! Saved to: fake_audio_english.wav


----------------------------------------


Targeting model: facebook/mms-tts-fra...


... (23 outputs left out)

Targeting model: facebook/mms-tts-kin...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

✅ Success! Saved to: fake_audio_kinyarwanda.wav


----------------------------------------


In [ ]:
!pip install -q transformers scipy gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 6.3 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import scipy.io.wavfile
import gc
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display, HTML
from gtts import gTTS

# 1. Hybrid Language Configuration
# 'mms' for Meta's Massively Multilingual Speech, 'gtts' for Google TTS fallback
LANG_CONFIG = {
    'English': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-eng'},
    'French': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-fra'},
    'German': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-deu'},
    'Spanish': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-spa'},
    'Chinese': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-zho'}, # Fixed to 'zho'
    'Catalan': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-cat'},
    'Bengali': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-ben'},
    'Kinyarwanda': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-kin'},

    # Using gTTS for languages lacking official MMS-TTS Hugging Face models
    'Esperanto': {'engine': 'gtts', 'gtts_code': 'eo'},
    'Belarusian': {'engine': 'gtts', 'gtts_code': 'be'},
    'Pashto': {'engine': 'gtts', 'gtts_code': 'ps'}
}

# 2. Sample Sentences
sentences = {
    "French": "Ceci est une voix générée artificiellement.",
    "German": "Dies ist eine künstlich erzeugte Stimme.",
    "Spanish": "Esta es una voz generada artificialmente.",
    "English": "This is an artificially generated voice.",
    "Kinyarwanda": "Ubu ni ijwi ryakozwe mu buryo bw'ubukorikori.",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo.",
    "Catalan": "Aquesta és una veu generada artificialment.",
    "Chinese": "这是由模型人工生成的语音。",
    "Bengali": "এটি কৃত্রিমভাবে তৈরি একটি কণ্ঠস্বর।",
    "Pashto": "دا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас."
}

def generate_audio_batch():
    """Generates audio using MMS or falls back to gTTS for missing languages."""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device.upper()}\n" + "="*40)

    for lang_name, lang_info in LANG_CONFIG.items():
        text = sentences.get(lang_name)
        if not text:
            continue

        display(HTML(f"<h3>Processing {lang_name}</h3>"))

        try:
            # --- META MMS ENGINE ---
            if lang_info['engine'] == 'mms':
                model_id = lang_info['mms_id']
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.wav")
                print(f"Targeting MMS model: {model_id}...")

                tokenizer = AutoTokenizer.from_pretrained(model_id)
                model = VitsModel.from_pretrained(model_id).to(device)
                inputs = tokenizer(text, return_tensors="pt").to(device)

                with torch.no_grad():
                    output = model(**inputs).waveform

                audio_data = output.squeeze().cpu().numpy()
                scipy.io.wavfile.write(output_filename, rate=model.config.sampling_rate, data=audio_data)

                # Clear memory immediately to prevent Colab from crashing
                del model, tokenizer, inputs, output
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # --- GOOGLE TTS ENGINE (FALLBACK) ---
            elif lang_info['engine'] == 'gtts':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.mp3")
                print(f"MMS unsupported for {lang_name}. Using Google TTS fallback...")

                tts = gTTS(text=text, lang=lang_info['gtts_code'])
                tts.save(output_filename)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed to process {lang_name}. Error: {e}")

        print("-" * 40)

# Run the generation
generate_audio_batch()

Using device: CUDA


Targeting MMS model: facebook/mms-tts-eng...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

✅ Success! Saved to: fake_audio_english.wav


----------------------------------------


Targeting MMS model: facebook/mms-tts-fra...


... (23 outputs left out)

----------------------------------------


MMS unsupported for Esperanto. Using Google TTS fallback...
❌ Failed to process Esperanto. Error: Language not supported: eo
----------------------------------------


MMS unsupported for Belarusian. Using Google TTS fallback...
❌ Failed to process Belarusian. Error: Language not supported: be
----------------------------------------


MMS unsupported for Pashto. Using Google TTS fallback...
❌ Failed to process Pashto. Error: Language not supported: ps
----------------------------------------


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Install required packages (Run this cell first)
!pip install -q edge-tts
!apt-get update -qq && apt-get install -qq espeak-ng

import edge_tts
import subprocess
from IPython.display import Audio, display, HTML

# 2. Configuration tailored for the 4 challenging languages
LANG_CONFIG = {
    'Chinese': {'engine': 'edge', 'voice': 'zh-CN-XiaoxiaoNeural'},     # Microsoft Azure Neural Female
    'Pashto': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural'},        # Microsoft Azure Neural Female
    'Belarusian': {'engine': 'edge', 'voice': 'be-BY-MaryiaNeural'},    # Microsoft Azure Neural Female
    'Esperanto': {'engine': 'espeak', 'voice': 'eo'}                    # Open Source Synthesizer
}

sentences = {
    "Chinese": "这是由模型人工生成的语音。",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo.",
    "Pashto": "دا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас."
}

async def generate_missing_languages():
    print("Initializing Audio Generation...\n" + "="*40)

    for lang_name, lang_info in LANG_CONFIG.items():
        text = sentences.get(lang_name)
        if not text:
            continue

        display(HTML(f"<h3>Processing {lang_name}</h3>"))

        try:
            # --- MICROSOFT EDGE NEURAL ENGINE ---
            if lang_info['engine'] == 'edge':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.mp3")
                print(f"Engine: Edge-TTS (Neural) | Voice: {lang_info['voice']}")

                communicate = edge_tts.Communicate(text, lang_info['voice'])
                await communicate.save(output_filename) # Awaits the file save

            # --- ESPEAK-NG ENGINE (ESPERANTO) ---
            elif lang_info['engine'] == 'espeak':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.wav")
                print(f"Engine: eSpeak-NG (Synthesizer) | Voice: {lang_info['voice']}")

                # Calls the underlying linux synthesizer installed via apt-get
                subprocess.run(['espeak-ng', '-v', lang_info['voice'], '-w', output_filename, text], check=True)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed to process {lang_name}. Error: {e}")

        print("-" * 40)

# 3. Execute the function
# (Colab Jupyter notebooks natively support awaiting async functions)
await generate_missing_languages()

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2

Engine: Edge-TTS (Neural) | Voice: zh-CN-XiaoxiaoNeural
✅ Success! Saved to: fake_audio_chinese.mp3


----------------------------------------


Engine: Edge-TTS (Neural) | Voice: ps-AF-LatifaNeural
✅ Success! Saved to: fake_audio_pashto.mp3


----------------------------------------


Engine: Edge-TTS (Neural) | Voice: be-BY-MaryiaNeural
❌ Failed to process Belarusian. Error: No audio was received. Please verify that your parameters are correct.
----------------------------------------


Engine: eSpeak-NG (Synthesizer) | Voice: eo
✅ Success! Saved to: fake_audio_esperanto.wav


----------------------------------------


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import edge_tts
import subprocess
from IPython.display import Audio, display, HTML

# FIXED: Route Belarusian to espeak-ng instead of edge-tts
LANG_CONFIG = {
    'Chinese': {'engine': 'edge', 'voice': 'zh-CN-XiaoxiaoNeural'},
    'Pashto': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural'},
    'Belarusian': {'engine': 'espeak', 'voice': 'be'},                  # Switched to eSpeak
    'Esperanto': {'engine': 'espeak', 'voice': 'eo'}
}

sentences = {
    "Chinese": "这是由模型人工生成的语音。",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo.",
    "Pashto": "دا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас."
}

async def generate_missing_languages():
    print("Initializing Audio Generation...\n" + "="*40)

    for lang_name, lang_info in LANG_CONFIG.items():
        text = sentences.get(lang_name)
        if not text:
            continue

        display(HTML(f"<h3>Processing {lang_name}</h3>"))

        try:
            if lang_info['engine'] == 'edge':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.mp3")
                print(f"Engine: Edge-TTS (Neural) | Voice: {lang_info['voice']}")

                communicate = edge_tts.Communicate(text, lang_info['voice'])
                await communicate.save(output_filename)

            elif lang_info['engine'] == 'espeak':
                # eSpeak natively outputs to WAV
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.wav")
                print(f"Engine: eSpeak-NG (Synthesizer) | Voice: {lang_info['voice']}")

                subprocess.run(['espeak-ng', '-v', lang_info['voice'], '-w', output_filename, text], check=True)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed to process {lang_name}. Error: {e}")

        print("-" * 40)

# Run the fixed script
await generate_missing_languages()

Initializing Audio Generation...


Engine: Edge-TTS (Neural) | Voice: zh-CN-XiaoxiaoNeural
✅ Success! Saved to: fake_audio_chinese.mp3


----------------------------------------


Engine: Edge-TTS (Neural) | Voice: ps-AF-LatifaNeural
✅ Success! Saved to: fake_audio_pashto.mp3


----------------------------------------


Engine: eSpeak-NG (Synthesizer) | Voice: be
❌ Failed to process Belarusian. Error: Command '['espeak-ng', '-v', 'be', '-w', 'fake_audio_belarusian.wav', 'Гэта штучна створаны голас.']' returned non-zero exit status 1.
----------------------------------------


Engine: eSpeak-NG (Synthesizer) | Voice: eo
✅ Success! Saved to: fake_audio_esperanto.wav


----------------------------------------


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import edge_tts
import subprocess
from IPython.display import Audio, display, HTML

# FIXED: Cross-Lingual workaround for Belarusian using a Russian Neural Voice
LANG_CONFIG = {
    'Chinese': {'engine': 'edge', 'voice': 'zh-CN-XiaoxiaoNeural'},
    'Pashto': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural'},
    'Esperanto': {'engine': 'espeak', 'voice': 'eo'},
    'Belarusian': {'engine': 'edge', 'voice': 'ru-RU-SvetlanaNeural'}   # Cyrillic phonetic workaround
}

sentences = {
    "Chinese": "这是由模型人工生成的语音。",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo.",
    "Pashto": "دا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас."
}

async def generate_missing_languages():
    print("Initializing Audio Generation...\n" + "="*40)

    for lang_name, lang_info in LANG_CONFIG.items():
        text = sentences.get(lang_name)
        if not text:
            continue

        display(HTML(f"<h3>Processing {lang_name}</h3>"))

        try:
            if lang_info['engine'] == 'edge':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.mp3")
                print(f"Engine: Edge-TTS (Neural) | Voice: {lang_info['voice']}")

                communicate = edge_tts.Communicate(text, lang_info['voice'])
                await communicate.save(output_filename)

            elif lang_info['engine'] == 'espeak':
                output_filename = str(OUTPUT_DIR / f"fake_audio_{lang_name.lower()}.wav")
                print(f"Engine: eSpeak-NG (Synthesizer) | Voice: {lang_info['voice']}")

                # Calling the underlying linux synthesizer installed via apt-get
                subprocess.run(['espeak-ng', '-v', lang_info['voice'], '-w', output_filename, text], check=True)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed to process {lang_name}. Error: {e}")

        print("-" * 40)

# Execute the function
await generate_missing_languages()

Initializing Audio Generation...


Engine: Edge-TTS (Neural) | Voice: zh-CN-XiaoxiaoNeural
✅ Success! Saved to: fake_audio_chinese.mp3


----------------------------------------


Engine: Edge-TTS (Neural) | Voice: ps-AF-LatifaNeural
✅ Success! Saved to: fake_audio_pashto.mp3


----------------------------------------


Engine: eSpeak-NG (Synthesizer) | Voice: eo
✅ Success! Saved to: fake_audio_esperanto.wav


----------------------------------------


Engine: Edge-TTS (Neural) | Voice: ru-RU-SvetlanaNeural
✅ Success! Saved to: fake_audio_belarusian.mp3


----------------------------------------


In [ ]:
!pip install -q coqui-tts transformers scipy edge-tts
!apt-get update -qq && apt-get install -qq espeak-ng

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 8.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch http

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import scipy.io.wavfile
import gc
import os
import subprocess
import edge_tts
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display, HTML

# Auto-accept Coqui XTTS Terms of Service
os.environ["COQUI_TOS_AGREED"] = "1"

# 1. The Ultimate Multi-Engine Configuration
LANG_CONFIG = {
    # --- XTTS-v2 Group (Hyper-Realistic) ---
    'English': {'engine': 'xtts', 'lang_code': 'en'},
    'French': {'engine': 'xtts', 'lang_code': 'fr'},
    'German': {'engine': 'xtts', 'lang_code': 'de'},
    'Spanish': {'engine': 'xtts', 'lang_code': 'es'},
    'Chinese': {'engine': 'xtts', 'lang_code': 'zh-cn'},

    # --- VITS / MMS Group (Massively Multilingual) ---
    'Catalan': {'engine': 'vits', 'mms_id': 'facebook/mms-tts-cat'},
    'Bengali': {'engine': 'vits', 'mms_id': 'facebook/mms-tts-ben'},
    'Kinyarwanda': {'engine': 'vits', 'mms_id': 'facebook/mms-tts-kin'},

    # --- Fallback Group (Unsupported by XTTS/VITS) ---
    'Pashto': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural'},
    'Belarusian': {'engine': 'edge', 'voice': 'ru-RU-SvetlanaNeural'}, # Cyrillic workaround
    'Esperanto': {'engine': 'espeak', 'voice': 'eo'}
}

# 2. Sentences
sentences = {
    "French": "Ceci est une voix générée artificiellement par XTTS.",
    "German": "Dies ist eine künstlich erzeugte Stimme.",
    "Spanish": "Esta es una voz generada artificialmente.",
    "English": "That's even how your millennial sibling talks about their childhood.",
    "Chinese": "这是由模型人工生成的语音。",
    "Catalan": "Aquesta és una veu generada artificialment.",
    "Bengali": "এটি কৃত্রিমভাবে তৈরি একটি কণ্ঠস্বর।",
    "Kinyarwanda": "Ubu ni ijwi ryakozwe mu buryo bw'ubukorikori.",
    "Esperanto": "Ĉi tio estas artefarite generita voĉo.",
    "Pashto": "دا په مصنوعي ډول رامینځته شوی غږ دی.",
    "Belarusian": "Гэта штучна створаны голас."
}

def clear_vram():
    """Forces Colab to release GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

async def generate_audio_master():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Initializing Master Audio Generation on: {device.upper()}\n" + "="*50)

    # ==========================================
    # PHASE 1: XTTS-v2 (Requires heavy VRAM)
    # ==========================================
    xtts_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] == 'xtts'}
    if xtts_langs:
        print("\n🚀 LOADING XTTS-v2 MODEL (~2GB VRAM)...")
        from TTS.api import TTS
        # Load model once for all XTTS languages
        xtts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

        for lang_name, lang_info in xtts_langs.items():
            text = sentences.get(lang_name)
            output_filename = str(OUTPUT_DIR / f"fake_audio_xtts_{lang_name.lower()}.wav")
            display(HTML(f"<h3>Processing {lang_name} (Engine: XTTS)</h3>"))

            try:
                # XTTS uses built-in default speakers. We'll use 'Ana Florence' as the universal voice donor
                xtts_model.tts_to_file(
                    text=text,
                    speaker="Ana Florence",
                    language=lang_info['lang_code'],
                    file_path=output_filename
                )
                print(f"✅ Success! Saved to: {output_filename}")
                display(Audio(output_filename))
            except Exception as e:
                print(f"❌ Failed XTTS {lang_name}: {e}")

        # Delete XTTS from memory before moving to VITS
        print("Unloading XTTS from VRAM to make space for VITS...")
        del xtts_model
        clear_vram()

    # ==========================================
    # PHASE 2: VITS / MMS
    # ==========================================
    vits_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] == 'vits'}
    for lang_name, lang_info in vits_langs.items():
        text = sentences.get(lang_name)
        output_filename = str(OUTPUT_DIR / f"fake_audio_vits_{lang_name.lower()}.wav")
        model_id = lang_info['mms_id']
        display(HTML(f"<h3>Processing {lang_name} (Engine: VITS/MMS)</h3>"))

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = VitsModel.from_pretrained(model_id).to(device)
            inputs = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                output = model(**inputs).waveform

            audio_data = output.squeeze().cpu().numpy()
            scipy.io.wavfile.write(output_filename, rate=model.config.sampling_rate, data=audio_data)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed VITS {lang_name}: {e}")
        finally:
            # Clear memory immediately after each VITS language
            if 'model' in locals(): del model
            if 'tokenizer' in locals(): del tokenizer
            clear_vram()

    # ==========================================
    # PHASE 3: EDGE / ESPEAK FALLBACKS
    # ==========================================
    fallback_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] in ['edge', 'espeak']}
    for lang_name, lang_info in fallback_langs.items():
        text = sentences.get(lang_name)
        display(HTML(f"<h3>Processing {lang_name} (Engine: {lang_info['engine'].upper()})</h3>"))

        try:
            if lang_info['engine'] == 'edge':
                output_filename = str(OUTPUT_DIR / f"fake_audio_edge_{lang_name.lower()}.mp3")
                communicate = edge_tts.Communicate(text, lang_info['voice'])
                await communicate.save(output_filename)

            elif lang_info['engine'] == 'espeak':
                output_filename = str(OUTPUT_DIR / f"fake_audio_espeak_{lang_name.lower()}.wav")
                subprocess.run(['espeak-ng', '-v', lang_info['voice'], '-w', output_filename, text], check=True)

            print(f"✅ Success! Saved to: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed Fallback {lang_name}: {e}")

    print("\n" + "="*50 + "\nAll 11 Languages Processed Successfully.")

# Run the master function
await generate_audio_master()

Initializing Master Audio Generation on: CUDA

🚀 LOADING XTTS-v2 MODEL (~2GB VRAM)...


✅ Success! Saved to: fake_audio_xtts_english.wav


✅ Success! Saved to: fake_audio_xtts_french.wav


✅ Success! Saved to: fake_audio_xtts_german.wav


✅ Success! Saved to: fake_audio_xtts_spanish.wav


❌ Failed XTTS Chinese: Chinese requires: pypinyin
Unloading XTTS from VRAM to make space for VITS...


... (6 outputs left out)

✅ Success! Saved to: fake_audio_edge_pashto.mp3


✅ Success! Saved to: fake_audio_edge_belarusian.mp3


✅ Success! Saved to: fake_audio_espeak_esperanto.wav



All 11 Languages Processed Successfully.


In [ ]:
!pip install -q coqui-tts transformers scipy edge-tts

In [ ]:
from pathlib import Path
OUTPUT_DIR = Path("tts_test_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import scipy.io.wavfile
import gc
import os
import edge_tts
from transformers import VitsModel, AutoTokenizer
from IPython.display import Audio, display, HTML

# Auto-accept Coqui XTTS Terms of Service
os.environ["COQUI_TOS_AGREED"] = "1"

# 1. Configuration matching your Image exactly + Dialect Mapping
LANG_CONFIG = {
    # --- x-TTS Group ---
    'eng': {'engine': 'xtts', 'lang_code': 'en', 'display': 'English'},
    'fra': {'engine': 'xtts', 'lang_code': 'fr', 'display': 'French'},
    'deu': {'engine': 'xtts', 'lang_code': 'de', 'display': 'German'},
    'spa': {'engine': 'xtts', 'lang_code': 'es', 'display': 'Spanish'},

    # --- MMS Group ---
    'cat': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-cat', 'display': 'Catalan'},
    'ben': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-ben', 'display': 'Bengali'},
    'kin': {'engine': 'mms', 'mms_id': 'facebook/mms-tts-kin', 'display': 'Kinyarwanda'},

    # --- Edge-TTS Group (Azure Neural) ---
    # Chinese and its dialects mapped directly to Azure per the screenshot
    'zho': {'engine': 'edge', 'voice': 'zh-CN-XiaoxiaoNeural', 'display': 'Chinese (zho)'},
    'cmn': {'engine': 'edge', 'voice': 'zh-CN-XiaoxiaoNeural', 'display': 'Chinese (cmn)'},

    # Pashto and its regional dialects mapped to Azure per the screenshot
    'pus': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural', 'display': 'Pashto (pus)'},
    'pbt': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural', 'display': 'Pashto (pbt)'},
    'pbu': {'engine': 'edge', 'voice': 'ps-AF-LatifaNeural', 'display': 'Pashto (pbu)'}

    # NOTE: Belarusian and Esperanto have been entirely excluded.
}

# 2. Sample Sentences mapped by Dialect ID
sentences = {
    "eng": "This is a fake audio sample in English.",
    "fra": "Ceci est un échantillon audio en français.",
    "deu": "Dies ist ein Audiobeispiel auf Deutsch.",
    "spa": "Esta es una muestra de audio en español.",
    "cat": "Aquesta és una mostra d'àudio en català.",
    "ben": "এটি বাংলায় একটি অডিও নমুনা।",
    "kin": "Uru ni urugero rw'amajwi mu Kinyarwanda.",
    "zho": "这是中文的音频样本 (zho).",
    "cmn": "这是普通话的音频样本 (cmn).",
    "pus": "دا په پښتو کې یو آډیو نمونه ده.",
    "pbt": "دا په سویلي پښتو کې یوه نمونه ده.", # Sample for Southern Pashto
    "pbu": "دا په شمالي پښتو کې یوه نمونه ده."  # Sample for Northern Pashto
}

def clear_vram():
    """Forces Colab to release GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

async def generate_audio_master():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Initializing Generation on: {device.upper()}\n" + "="*50)

    # ==========================================
    # PHASE 1: x-TTS (English, French, German, Spanish)
    # ==========================================
    xtts_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] == 'xtts'}
    if xtts_langs:
        print("\n🚀 LOADING x-TTS MODEL (~2GB VRAM)...")
        from TTS.api import TTS
        xtts_model = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

        for dialect_id, lang_info in xtts_langs.items():
            text = sentences.get(dialect_id)
            output_filename = str(OUTPUT_DIR / f"audio_{dialect_id}_{lang_info['engine']}.wav")
            display(HTML(f"<h3>Processing {lang_info['display']}</h3>"))

            try:
                xtts_model.tts_to_file(
                    text=text,
                    speaker="Ana Florence",
                    language=lang_info['lang_code'],
                    file_path=output_filename
                )
                print(f"✅ Saved: {output_filename}")
                display(Audio(output_filename))
            except Exception as e:
                print(f"❌ Failed {lang_info['display']}: {e}")

        # Free VRAM
        print("Unloading x-TTS from VRAM...")
        del xtts_model
        clear_vram()

    # ==========================================
    # PHASE 2: MMS (Catalan, Bengali, Kinyarwanda)
    # ==========================================
    print("\n🚀 LOADING MMS MODELS...")
    vits_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] == 'mms'}
    for dialect_id, lang_info in vits_langs.items():
        text = sentences.get(dialect_id)
        output_filename = str(OUTPUT_DIR / f"audio_{dialect_id}_{lang_info['engine']}.wav")
        model_id = lang_info['mms_id']
        display(HTML(f"<h3>Processing {lang_info['display']}</h3>"))

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = VitsModel.from_pretrained(model_id).to(device)
            inputs = tokenizer(text, return_tensors="pt").to(device)

            with torch.no_grad():
                output = model(**inputs).waveform

            audio_data = output.squeeze().cpu().numpy()
            scipy.io.wavfile.write(output_filename, rate=model.config.sampling_rate, data=audio_data)
            print(f"✅ Saved: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed {lang_info['display']}: {e}")
        finally:
            if 'model' in locals(): del model
            if 'tokenizer' in locals(): del tokenizer
            clear_vram()

    # ==========================================
    # PHASE 3: EDGE-TTS (Chinese & Pashto Dialects)
    # ==========================================
    print("\n🚀 INITIALIZING EDGE-TTS ENGINE...")
    edge_langs = {k: v for k, v in LANG_CONFIG.items() if v['engine'] == 'edge'}
    for dialect_id, lang_info in edge_langs.items():
        text = sentences.get(dialect_id)
        display(HTML(f"<h3>Processing {lang_info['display']}</h3>"))

        try:
            output_filename = str(OUTPUT_DIR / f"audio_{dialect_id}_{lang_info['engine']}.mp3")
            communicate = edge_tts.Communicate(text, lang_info['voice'])
            await communicate.save(output_filename)

            print(f"✅ Saved: {output_filename}")
            display(Audio(output_filename))

        except Exception as e:
            print(f"❌ Failed {lang_info['display']}: {e}")

    print("\n" + "="*50 + "\nAll Target Dialects Processed Successfully.")

# Run the master function
await generate_audio_master()

Initializing Generation on: CUDA

🚀 LOADING x-TTS MODEL (~2GB VRAM)...


✅ Saved: audio_eng_xtts.wav


✅ Saved: audio_fra_xtts.wav


✅ Saved: audio_deu_xtts.wav


✅ Saved: audio_spa_xtts.wav


Unloading x-TTS from VRAM...

🚀 LOADING MMS MODELS...


... (9 outputs left out)

✅ Saved: audio_pus_edge.mp3


✅ Saved: audio_pbt_edge.mp3


✅ Saved: audio_pbu_edge.mp3



All Target Dialects Processed Successfully.


RuntimeError: Set OPENAI_API_KEY in your environment first.